In [278]:
import numpy as np
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [279]:
with open('intents.json', 'r') as file:
    data = json.load(file)

print(data)

{'intents': [{'tag': 'greeting', 'patterns': ['Hi', 'Hey', 'Hello', 'Good morning', 'Good afternoon', 'Is anyone available?'], 'responses': ['Hello! Welcome to Hearth & Beans Support. How can I assist you today?', 'Hi there! How can I help you with your order or inquiry today?']}, {'tag': 'goodbye', 'patterns': ['Bye', 'Goodbye', 'Have a good day', 'See you'], 'responses': ['Thank you for contacting Hearth & Beans Support. Have a great day!', 'Goodbye! Feel free to reach out if you need anything else.']}, {'tag': 'thanks', 'patterns': ['Thanks', 'Thank you', 'That helped', 'Thanks for your help'], 'responses': ["You're very welcome! Let me know if you need help with anything else.", 'Glad I could help! Enjoy your coffee.']}, {'tag': 'menu_and_items', 'patterns': ['What do you sell?', 'Can I see the menu?', 'What drinks do you have?', 'Do you have snacks or pastries?'], 'responses': ['We serve hot and cold coffees, teas, hot chocolate, and fresh pastries/baked goods. You can view our fu

In [280]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Text Preprocessing

In [281]:
import nltk
print(nltk.__file__)

/usr/local/lib/python3.13/dist-packages/nltk/__init__.py


In [282]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [283]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [284]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [285]:
def preprocess_text(text):
    # convert to lowercase
    text = str(text).lower()
    # remove punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    words = text.split()
    # remove stopwords
    words = [word for word in words if word not in stop_words]
    # lemmatize (lemmatize means changing a word back to its base form. ["walking" => walk])
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)


In [286]:
all_words = []
unique_tags = []
patterns_and_tags = []

In [287]:
for intent in data['intents']:
    tag = intent['tag']
    unique_tags.append(tag)

    for pattern in intent['patterns']:
        preprocessed_pattern = preprocess_text(pattern)
        all_words.extend(preprocessed_pattern.split())

        patterns_and_tags.append((preprocessed_pattern, tag))

all_words = sorted(set(all_words))
unique_tags = sorted(set(unique_tags))

In [288]:
patterns_and_tags

[('hi', 'greeting'),
 ('hey', 'greeting'),
 ('hello', 'greeting'),
 ('good morning', 'greeting'),
 ('good afternoon', 'greeting'),
 ('anyone available', 'greeting'),
 ('bye', 'goodbye'),
 ('goodbye', 'goodbye'),
 ('good day', 'goodbye'),
 ('see', 'goodbye'),
 ('thanks', 'thanks'),
 ('thank', 'thanks'),
 ('helped', 'thanks'),
 ('thanks help', 'thanks'),
 ('sell', 'menu_and_items'),
 ('see menu', 'menu_and_items'),
 ('drink', 'menu_and_items'),
 ('snack pastry', 'menu_and_items'),
 ('opening hour', 'store_timings'),
 ('cafe open', 'store_timings'),
 ('open weekend', 'store_timings'),
 ('time close', 'store_timings'),
 ('located', 'location'),
 ('address', 'location'),
 ('get hearth bean', 'location'),
 ('cafe', 'location'),
 ('order', 'order_status'),
 ('track order', 'order_status'),
 ('long order take', 'order_status'),
 ('order delayed', 'order_status'),
 ('order wrong', 'order_issue'),
 ('missing item order', 'order_issue'),
 ('got wrong coffee', 'order_issue'),
 ('drink cold', 'orde

In [289]:
preprocessed_sentences = [pattern for (pattern, tag) in patterns_and_tags]
training_labels = [tag for (pattern, tag) in patterns_and_tags]

## Feature Vectorisation

In [290]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(preprocessed_sentences).toarray()

In [291]:
y_train = []
for tag in training_labels:
    y_train.append(unique_tags.index(tag))

In [292]:
len(y_train)

2830

In [293]:
epochs = 1000
batch_size = 8
learning_rate = 0.001
input_size = X_train.shape[1]
hidden_size = 32
output_size = len(unique_tags)

In [294]:
input_size, output_size

(2536, 935)

**Dataset object to convert numpy array to tensor**

In [295]:
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = torch.from_numpy(X_train)
        self.y_data = torch.from_numpy(np.array(y_train))

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

In [296]:
train_loader = DataLoader(dataset=ChatDataset(), batch_size=batch_size, shuffle=True)

In [297]:
class ChatbotModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ChatbotModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=0.02),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=0.02),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.model(x)


In [298]:
model = ChatbotModel(input_size, hidden_size, output_size).to(device)

In [299]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [275]:
for epoch in range(epochs):
    total_train_loss = 0
    total_train_accuracy = 0

    for (words, targets) in train_loader:
        words = words.to(device).float()
        targets = targets.to(device)

        # forward pass
        outputs = model(words)
        loss = criterion(outputs, targets)

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        # calculate multi-class accuracy using argmax
        predictions = torch.argmax(outputs, dim=1)
        total_train_accuracy += (predictions == targets).sum().item()

        epoch_loss = total_train_loss / len(train_loader)

    # print metrics every 100 epochs
    if (epoch + 1) % 100 == 0:
        train_loss = total_train_loss / len(train_loader)
        train_accuracy = (total_train_accuracy / len(X_train)) * 100
        print(f"| Epoch: {epoch + 1}/{epochs} | Train Accuracy: {train_accuracy:.2f}% | Train Loss: {train_loss:.4f} |")

| Epoch: 100/1000 | Train Accuracy: 98.83% | Train Loss: 0.0355 |
| Epoch: 200/1000 | Train Accuracy: 99.58% | Train Loss: 0.0121 |
| Epoch: 300/1000 | Train Accuracy: 99.29% | Train Loss: 0.0141 |
| Epoch: 400/1000 | Train Accuracy: 99.72% | Train Loss: 0.0059 |
| Epoch: 500/1000 | Train Accuracy: 99.82% | Train Loss: 0.0034 |
| Epoch: 600/1000 | Train Accuracy: 99.79% | Train Loss: 0.0032 |
| Epoch: 700/1000 | Train Accuracy: 99.72% | Train Loss: 0.0056 |
| Epoch: 800/1000 | Train Accuracy: 99.79% | Train Loss: 0.0052 |
| Epoch: 900/1000 | Train Accuracy: 99.68% | Train Loss: 0.0053 |
| Epoch: 1000/1000 | Train Accuracy: 99.65% | Train Loss: 0.0151 |


In [303]:
final_train_loss = total_train_loss / len(train_loader)
final_train_accuracy = (total_train_accuracy / len(X_train)) * 100

print(f"Final Training Accuracy: {final_train_accuracy:.2f}%")
print(f"Final Training Loss: {final_train_loss:.4f}")

Final Training Accuracy: 99.65%
Final Training Loss: 0.0151


In [304]:
import torch
import joblib

torch.save(model.state_dict(), "chatbot.pth")
joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(unique_tags, "tags.pkl")

['tags.pkl']